In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('../atp/data/tennis.db')

df = pd.read_sql_query("""
    SELECT
        m.id,
        m.match_code,
        m.reason,
        m.number_of_sets,
        m.p1_score,
        m.p2_score,
        t.event_year,
        t.tournament_name
    FROM matches m
    JOIN tournaments t ON m.tournament_id = t.id
    WHERE m.p1_score IS NOT NULL
      AND m.p2_score IS NOT NULL
      AND m.number_of_sets IS NOT NULL
""", conn)

conn.close()
print(f'Total matches: {len(df)}')
print(f'Retirements (reason=RET): {(df.reason == "RET").sum()}')
df.head()

In [ ]:
def is_valid_set(a, b):
    high, low = max(a, b), min(a, b)
    if high == 6 and low <= 4: return True
    if high == 7 and low in (5, 6): return True
    return False

def is_retirement_valid(sets, best_of):
    """Returns True if scores indicate a legitimate incomplete match (retirement),
    False if the match appears to have completed normally (data error)."""
    sets_needed = best_of // 2 + 1
    wins_a = wins_b = 0
    for i, (a, b) in enumerate(sets):
        if not is_valid_set(a, b):
            return True   # incomplete set = valid retirement
        if a > b: wins_a += 1
        else: wins_b += 1
        if max(wins_a, wins_b) >= sets_needed:
            if i < len(sets) - 1:
                return False  # more sets after match decided = data error
            return False      # match completed normally
    return True  # no winner reached = valid retirement

def classify_match(row):
    try:
        p1 = [int(x) for x in str(row['p1_score']).split(';')]
        p2 = [int(x) for x in str(row['p2_score']).split(';')]
        sets = list(zip(p1, p2))
        return is_retirement_valid(sets, best_of=int(row['number_of_sets']))
    except Exception:
        return None  # unparseable

df['score_incomplete'] = df.apply(classify_match, axis=1)
df['flagged_ret'] = df['reason'] == 'RET'
print('Done classifying')

In [ ]:
# Consistency check
# score_incomplete=True  + flagged_ret=True  -> consistent retirement
# score_incomplete=False + flagged_ret=False -> consistent normal match
# score_incomplete=True  + flagged_ret=False -> score looks incomplete but no RET flag
# score_incomplete=False + flagged_ret=True  -> RET flag but score looks complete (data error)

consistent_ret     = df[ df.score_incomplete &  df.flagged_ret]
consistent_normal  = df[~df.score_incomplete & ~df.flagged_ret]
incomplete_no_flag = df[ df.score_incomplete & ~df.flagged_ret]
complete_but_ret   = df[~df.score_incomplete &  df.flagged_ret]

print(f'Consistent retirements (score incomplete + RET flag):    {len(consistent_ret)}')
print(f'Consistent normal matches (score complete + no RET flag): {len(consistent_normal)}')
print(f'Score incomplete but no RET flag:                         {len(incomplete_no_flag)}')
print(f'RET flag but score looks complete (data errors):          {len(complete_but_ret)}')

In [ ]:
# Inspect: score looks complete but tagged RET
cols = ['event_year', 'tournament_name', 'match_code', 'number_of_sets', 'p1_score', 'p2_score', 'reason']
print('=== RET flag but score appears complete ===')
display(complete_but_ret[cols])

In [ ]:
# Inspect: score incomplete but no RET flag
print('=== Score incomplete but no RET flag ===')
display(incomplete_no_flag[cols])